# AI-Assisted Data Cleaning & Optimization — Online Retail Dataset

**Project goal:** assess data quality, remove exact duplicate records, handle only safely inferable missing values, review unusual transaction values, and optimize data types without changing the business meaning of the records.

## Important decisions for this dataset

- `InvoiceNo`, `StockCode`, and `CustomerID` are identifiers/categories, not continuous numerical measurements.
- An invoice number beginning with `C` indicates a cancellation. Negative `Quantity` values can represent returns/cancellations; they must not be deleted just because they are negative.
- `CustomerID` is missing on many records. A missing customer identity cannot be inferred safely from the other columns, so it is kept missing rather than replaced with a fake ID such as `-1`.
- `Description` can be inferred from `StockCode` only when that stock code has exactly one distinct non-missing description in the data. Ambiguous or unknown descriptions remain missing.
- IQR outliers are flagged for investigation only. This notebook does **not** cap or delete outliers automatically.
- Exact duplicates are removed from the raw data **before** imputation so that imputation cannot create records that are then mistakenly treated as original duplicates.



## 1. Import dependencies

### AI prompt for this code

> I am building an AI-assisted Python data-cleaning project for the UCI Online Retail dataset. Write the setup cell for a Jupyter Notebook. Import the standard libraries needed for file paths, tabular data handling, and numerical operations (`pathlib`, `pandas`, and `numpy`). Also import the plotting libraries already planned for this notebook (`matplotlib.pyplot` and `seaborn`). Print the installed pandas and NumPy versions and a short project-ready welcome message. Keep the cell simple, readable, and compatible with a modern Jupyter environment. Do not load or modify the dataset in this cell.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Welcome to the Online Retail data-cleaning project!")

# For this .xlsx file, pandas uses openpyxl.
# xlrd is only needed for older .xls files, not this .xlsx workbook.

Pandas: 3.1.1
NumPy: 2.5.4
Welcome to the Online Retail data-cleaning project!


## 2. Load the dataset

### AI prompt for this code

> Write a robust Jupyter Notebook cell to load the UCI Online Retail Excel workbook into a pandas DataFrame named `df`. Check a small list of likely file locations and filenames, including the notebook folder and a `data` subfolder. If no file is found, raise a helpful `FileNotFoundError` explaining how to fix the path. Read the `.xlsx` file with `openpyxl`, verify that all eight expected fields are present (`InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, and `Country`), and raise a clear error if any are missing. Reorder the columns consistently, then print the source filename and dataset shape and display the first five rows. Do not clean or transform values while loading.

In [2]:
# The notebook checks common file locations/names.
# Keep the Excel file in the same folder as this notebook, or update DATA_FILE.
possible_files = [
    Path("Online_Retail.xlsx"),
    Path("Online_Retail(1).xlsx"),
    Path("data/Online_Retail.xlsx"),
    Path("data/Online_Retail(1).xlsx"),
]

DATA_FILE = next((path for path in possible_files if path.exists()), None)

if DATA_FILE is None:
    raise FileNotFoundError(
        "Online Retail workbook not found. Place Online_Retail.xlsx "
        "in the notebook folder or update possible_files."
    )

df = pd.read_excel(DATA_FILE, engine="openpyxl")

required_columns = [
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country"
]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"The dataset is missing required columns: {missing_columns}")

# Keep the expected column order for consistent reporting.
df = df[required_columns].copy()

print("Loaded file:", DATA_FILE)
print("Dataset shape:", df.shape)
display(df.head())

Loaded file: Online_Retail.xlsx
Dataset shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 3. Initial dataset inspection

### AI prompt for this code

> Write a reusable `data_quality_assessment(data)` function for the UCI Online Retail DataFrame. Validate that the input is a non-empty pandas DataFrame. Return separate reports for (1) missing-value counts and percentages by column, (2) the number and percentage of exact duplicate rows, (3) IQR outlier flags for `Quantity` and `UnitPrice` only, and (4) retail-specific diagnostics such as cancellation invoices (`InvoiceNo` beginning with `C`), negative quantities, zero prices, and negative prices. Treat `CustomerID`, `InvoiceNo`, and `StockCode` as identifiers/categories, not continuous measures. Do not remove or alter any records. Also produce an overall summary report. Call the function on `df` and display all reports with clear labels. Use outlier flags as investigation indicators, not automatic deletion rules.

In [3]:
print("Shape:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nDescriptive summary:")
display(df.describe(include="all").T)

print("\nDate range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())

Shape: (541909, 8)

Data types:


,Data Type
InvoiceNo,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[us]
UnitPrice,float64
CustomerID,float64
Country,str



Descriptive summary:


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
InvoiceNo,541909.0,25900.0,573585.0,1114.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,541909,4070,85123A,2313,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,540455,4223,WHITE HANGING HEART T-LIGHT HOLDER,2369,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,541909.0,NaN,NaN,NaN,9.55225,-80995.0,1.0,3.0,10.0,80995.0,218.081158
InvoiceDate,541909,NaN,NaN,NaN,2011-07-04 13:34:57.156386,2010-12-01 08:26:00,2011-03-28 11:34:00,2011-07-19 17:17:00,2011-10-19 11:27:00,2011-12-09 12:50:00,NaN
UnitPrice,541909.0,NaN,NaN,NaN,4.611114,-11062.06,1.25,2.08,4.13,38970.0,96.759853
CustomerID,406829.0,NaN,NaN,NaN,15287.69057,12346.0,13953.0,15152.0,16791.0,18287.0,1713.600303
Country,541909,38,United Kingdom,495478,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


## 4. Data quality assessment

The IQR check is applied only to `Quantity` and `UnitPrice`. `CustomerID` is excluded because it is a nominal identifier, even though Excel/pandas initially represents it as a number. The additional business-rule checks are descriptive diagnostics only; they do not modify the data.

### AI prompt for this code

> Write a duplicate-review and removal cell for the original raw Online Retail DataFrame `df`, before any missing-value imputation. Identify rows that are identical across all eight columns, retain the first copy, and create `deduplicated_df`. Keep a separate copy of all rows participating in exact duplicates for review. Also count repeated `InvoiceNo` + `StockCode` combinations for information only; do not remove those combinations because a single invoice can contain multiple product lines. Create and display a concise duplicate summary and a sample of duplicate rows. Add assertions confirming the expected row-count change and that no exact duplicates remain. Do not use `InvoiceNo` alone as a unique row key.

In [4]:
def data_quality_assessment(data):
    """Return missing-value, duplicate, IQR-outlier and business-rule reports."""
    if not isinstance(data, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")
    if data.empty:
        raise ValueError("Input DataFrame is empty.")

    total_rows = len(data)
    missing_count = data.isna().sum()
    missing_report = pd.DataFrame({
        "Missing_Count": missing_count,
        "Missing_Percentage": (missing_count / total_rows * 100).round(2)
    }).sort_values("Missing_Percentage", ascending=False)

    duplicate_count = int(data.duplicated(keep="first").sum())
    duplicate_report = pd.DataFrame({
        "Metric": ["Total Records", "Exact Duplicate Rows To Remove", "Rows After Exact Deduplication", "Duplicate Percentage"],
        "Value": [total_rows, duplicate_count, total_rows - duplicate_count,
                  round(duplicate_count / total_rows * 100, 2)]
    })

    quantitative_columns = [c for c in ["Quantity", "UnitPrice"] if c in data.columns]
    outlier_rows = []
    for col in quantitative_columns:
        values = pd.to_numeric(data[col], errors="coerce").dropna()
        if values.empty:
            continue
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        mask = (values < lower) | (values > upper)
        outlier_rows.append({
            "Column": col,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "Lower_Bound": lower,
            "Upper_Bound": upper,
            "Outlier_Count": int(mask.sum()),
            "Outlier_Percentage": round(mask.mean() * 100, 2)
        })
    outlier_report = pd.DataFrame(outlier_rows)

    invoice_text = data["InvoiceNo"].astype("string") if "InvoiceNo" in data else pd.Series(dtype="string")
    business_rule_report = pd.DataFrame({
        "Check": [
            "InvoiceNo starts with C (cancellation marker)",
            "Negative Quantity rows (returns/cancellations may be present)",
            "Negative Quantity rows without C invoice prefix (review only)",
            "Zero UnitPrice rows (review only)",
            "Negative UnitPrice rows (review only)"
        ],
        "Count": [
            int(invoice_text.str.startswith("C", na=False).sum()),
            int((data["Quantity"] < 0).sum()),
            int(((data["Quantity"] < 0) & ~invoice_text.str.startswith("C", na=False)).sum()),
            int((data["UnitPrice"] == 0).sum()),
            int((data["UnitPrice"] < 0).sum())
        ]
    })

    total_missing = int(missing_count.sum())
    total_cells = int(data.size)
    summary_report = pd.DataFrame({
        "Metric": [
            "Total Rows", "Total Columns", "Total Missing Cells",
            "Missing Cell Percentage", "Columns With Missing Values",
            "Exact Duplicate Rows To Remove", "Quantitative Columns Assessed",
            "Total IQR Outlier Flags"
        ],
        "Value": [
            total_rows, data.shape[1], total_missing,
            round(total_missing / total_cells * 100, 2) if total_cells else 0,
            int((missing_count > 0).sum()), duplicate_count,
            len(quantitative_columns),
            int(outlier_report["Outlier_Count"].sum()) if not outlier_report.empty else 0
        ]
    })

    return missing_report, duplicate_report, outlier_report, business_rule_report, summary_report

missing_report, duplicate_report, outlier_report, business_rule_report, summary_report = data_quality_assessment(df)

print("MISSING VALUES")
display(missing_report)
print("DUPLICATES")
display(duplicate_report)
print("IQR OUTLIER FLAGS — INVESTIGATE, DO NOT DELETE AUTOMATICALLY")
display(outlier_report)
print("BUSINESS-RULE DIAGNOSTICS")
display(business_rule_report)
print("OVERALL SUMMARY")
display(summary_report)

MISSING VALUES


,Missing_Count,Missing_Percentage
CustomerID,135080,24.93
Description,1454,0.27
InvoiceNo,0,0.00
StockCode,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
Country,0,0.00


DUPLICATES


,Metric,Value
0,Total Records,541909.00
1,Exact Duplicate Rows To Remove,5268.00
2,Rows After Exact Deduplication,536641.00
3,Duplicate Percentage,0.97


IQR OUTLIER FLAGS — INVESTIGATE, DO NOT DELETE AUTOMATICALLY


,Column,Q1,Q3,IQR,Lower_Bound,Upper_Bound,Outlier_Count,Outlier_Percentage
0,Quantity,1.00,10.00,9.00,-12.50,23.50,58619,10.82
1,UnitPrice,1.25,4.13,2.88,-3.07,8.45,39627,7.31


BUSINESS-RULE DIAGNOSTICS


,Check,Count
0,InvoiceNo starts with C (cancellation marker),9288
1,Negative Quantity rows (returns/cancellations ...,10624
2,Negative Quantity rows without C invoice prefi...,1336
3,Zero UnitPrice rows (review only),2515
4,Negative UnitPrice rows (review only),2


OVERALL SUMMARY


,Metric,Value
0,Total Rows,541909.00
1,Total Columns,8.00
2,Total Missing Cells,136534.00
3,Missing Cell Percentage,3.15
4,Columns With Missing Values,2.00
5,Exact Duplicate Rows To Remove,5268.00
6,Quantitative Columns Assessed,2.00
7,Total IQR Outlier Flags,98246.00


## 5. Remove exact duplicate rows from the raw data

Only rows that match across **all eight columns** are removed. Repeated `InvoiceNo`, repeated `StockCode`, or repeated `InvoiceNo + StockCode` combinations are not automatically removed: invoices legitimately contain multiple product lines, and a repeated product can occur on separate line items.

### AI prompt for this code

> Write a duplicate-review and removal cell for the original raw Online Retail DataFrame `df`, before any missing-value imputation. Identify rows that are identical across all eight columns, retain the first copy, and create `deduplicated_df`. Keep a separate copy of all rows participating in exact duplicates for review. Also count repeated `InvoiceNo` + `StockCode` combinations for information only; do not remove those combinations because a single invoice can contain multiple product lines. Create and display a concise duplicate summary and a sample of duplicate rows. Add assertions confirming the expected row-count change and that no exact duplicates remain. Do not use `InvoiceNo` alone as a unique row key.

In [5]:
# Detect duplicates on the original raw dataset, before any imputation.
exact_duplicate_mask = df.duplicated(keep=False)
duplicate_rows_for_review = df.loc[exact_duplicate_mask].copy()
exact_duplicates_to_remove = int(df.duplicated(keep="first").sum())

# Repeated invoice/product combinations are reported for awareness only.
repeated_invoice_stockcode = int(
    df.duplicated(subset=["InvoiceNo", "StockCode"], keep="first").sum()
)

deduplicated_df = df.drop_duplicates(keep="first").copy()

# Compact duplicate summary.
duplicate_summary = pd.DataFrame({
    "Metric": [
        "Rows before deduplication",
        "Exact duplicate rows removed",
        "Rows after deduplication",
        "Remaining exact duplicates",
        "Repeated InvoiceNo + StockCode combinations (not removed)"
    ],
    "Value": [
        len(df), exact_duplicates_to_remove, len(deduplicated_df),
        int(deduplicated_df.duplicated().sum()), repeated_invoice_stockcode
    ]
})

duplicate_patterns = (
    duplicate_rows_for_review.value_counts(dropna=False)
    .rename("Occurrences")
    .reset_index()
    .sort_values("Occurrences", ascending=False)
)

display(duplicate_summary)
print("Sample exact duplicate rows:")
display(duplicate_rows_for_review.head(10))

assert len(deduplicated_df) == len(df) - exact_duplicates_to_remove
assert deduplicated_df.duplicated().sum() == 0
print("Exact duplicate removal validation passed.")

,Metric,Value
0,Rows before deduplication,541909
1,Exact duplicate rows removed,5268
2,Rows after deduplication,536641
3,Remaining exact duplicates,0
4,Repeated InvoiceNo + StockCode combinations (n...,10684


Sample exact duplicate rows:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
548,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom


Exact duplicate removal validation passed.


## 6. Missing-value handling

This section is deliberately conservative:

1. `CustomerID`: keep missing values. It is an identifier, and assigning `-1` would create a fake customer. The missingness itself should be documented.
2. `Description`: fill only when `StockCode` maps to exactly one distinct known description. If a code has conflicting descriptions or no known description, leave it missing.
3. Other categorical columns: use the mode only if a missing value actually exists.
4. Quantitative columns: mean for approximately symmetric data and median for skewed data only if missing values exist. In this particular workbook, `Quantity` and `UnitPrice` have no missing values, so this step should not change them.

### AI prompt for this code

> Create a final validation and reporting cell for the cleaned and optimized Online Retail DataFrame. Verify that optimization preserved the DataFrame shape, missing-value counts, duplicate count, and all column values (use appropriate comparisons or stable hashes for category/string columns). Explicitly check that cancellation invoice markers are unchanged, missing `CustomerID` values remain missing, `UnitPrice` is still `float64`, and `InvoiceDate` remains datetime. Build and display a final summary containing the original row count, post-deduplication row count, duplicates removed, remaining missing identifiers/descriptions, and memory before/after optimization and percentage reduction. Export the compact quality and optimization reports into one Excel workbook with clearly named sheets. Keep full cleaned-data CSV and Excel exports optional and disabled by default, because the dataset is large. Finish with clear validation messages.

In [6]:
SKEW_THRESHOLD = 0.5

def calculate_quality_metrics(data):
    total_cells = data.size
    total_missing = int(data.isna().sum().sum())
    numeric_columns = [c for c in ["Quantity", "UnitPrice"] if c in data.columns]
    outlier_total = 0
    for col in numeric_columns:
        values = pd.to_numeric(data[col], errors="coerce").dropna()
        if values.empty:
            continue
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        outlier_total += int(((values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)).sum())
    return {
        "Rows": len(data),
        "Columns": len(data.columns),
        "Missing_Cells": total_missing,
        "Missing_Cell_Percentage": round(total_missing / total_cells * 100, 2) if total_cells else 0,
        "Exact_Duplicates": int(data.duplicated().sum()),
        "IQR_Outlier_Flags_Quantity_UnitPrice": outlier_total
    }


def intelligent_imputation(data, skew_threshold=0.5):
    if not isinstance(data, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")
    if data.empty:
        raise ValueError("Input DataFrame is empty.")

    result = data.copy(deep=True)
    strategy_rows = []
    description_details = {"Missing_Before": 0, "Filled_From_StockCode": 0, "Remaining_Missing": 0,
                           "Conflicting_StockCodes_Not_Imputed": 0}

    before_metrics = calculate_quality_metrics(result)
    missing_before = result.isna().sum()

    # A. Keep CustomerID missing; do not create a synthetic customer identifier.
    if "CustomerID" in result.columns:
        customer_numeric = pd.to_numeric(result["CustomerID"], errors="coerce")
        non_missing_ids = customer_numeric.dropna()
        if not np.allclose(non_missing_ids, np.round(non_missing_ids)):
            raise ValueError("CustomerID contains non-integer values; inspect before converting.")
        result["CustomerID"] = customer_numeric.astype("Int32")
        strategy_rows.append({
            "Column": "CustomerID", "Strategy": "Not imputed — identifier; missing values retained",
            "Missing_Before": int(missing_before["CustomerID"]),
            "Filled": 0, "Missing_After": int(result["CustomerID"].isna().sum())
        })

    # B. Numeric columns: only measure columns, never IDs.
    numeric_columns = [c for c in ["Quantity", "UnitPrice"] if c in result.columns]
    for col in numeric_columns:
        missing_count = int(result[col].isna().sum())
        skewness = result[col].dropna().skew()
        if missing_count == 0:
            strategy_rows.append({
                "Column": col, "Strategy": "No imputation needed — no missing values",
                "Skewness": round(skewness, 3) if pd.notna(skewness) else np.nan,
                "Missing_Before": 0, "Filled": 0, "Missing_After": 0
            })
            continue
        strategy = "Mean" if pd.notna(skewness) and abs(skewness) <= skew_threshold else "Median"
        fill_value = result[col].mean() if strategy == "Mean" else result[col].median()
        result[col] = result[col].fillna(fill_value)
        strategy_rows.append({
            "Column": col, "Strategy": strategy,
            "Skewness": round(skewness, 3) if pd.notna(skewness) else np.nan,
            "Imputation_Value": fill_value,
            "Missing_Before": missing_count,
            "Filled": int(missing_count - result[col].isna().sum()),
            "Missing_After": int(result[col].isna().sum())
        })

    # C. Description: fill only unambiguous StockCode-to-Description mappings.
    if {"StockCode", "Description"}.issubset(result.columns):
        description_details["Missing_Before"] = int(result["Description"].isna().sum())
        valid_pairs = result[["StockCode", "Description"]].dropna()
        descriptions_per_code = valid_pairs.groupby("StockCode")["Description"].nunique()
        unambiguous_codes = descriptions_per_code[descriptions_per_code == 1].index
        conflicting_codes = descriptions_per_code[descriptions_per_code > 1].index
        mapping = (
            valid_pairs[valid_pairs["StockCode"].isin(unambiguous_codes)]
            .drop_duplicates(subset="StockCode")
            .set_index("StockCode")["Description"]
            .to_dict()
        )
        missing_mask = result["Description"].isna()
        inferred = result.loc[missing_mask, "StockCode"].map(mapping)
        filled_mask = missing_mask & inferred.notna()
        result.loc[filled_mask, "Description"] = inferred.loc[filled_mask]
        description_details["Filled_From_StockCode"] = int(filled_mask.sum())
        description_details["Remaining_Missing"] = int(result["Description"].isna().sum())
        description_details["Conflicting_StockCodes_Not_Imputed"] = int(
            result.loc[missing_mask & ~inferred.notna(), "StockCode"].isin(conflicting_codes).sum()
        )
        strategy_rows.append({
            "Column": "Description", "Strategy": "StockCode mapping only when mapping is unambiguous",
            "Missing_Before": description_details["Missing_Before"],
            "Filled": description_details["Filled_From_StockCode"],
            "Missing_After": description_details["Remaining_Missing"]
        })

    # D. Other nominal columns: mode only when missing values exist.
    excluded = {"InvoiceNo", "StockCode", "CustomerID", "Description", "InvoiceDate"}
    categorical_columns = [
        c for c in result.select_dtypes(include=["object", "category", "string"]).columns
        if c not in excluded
    ]
    for col in categorical_columns:
        missing_count = int(result[col].isna().sum())
        if missing_count == 0:
            strategy_rows.append({
                "Column": col, "Strategy": "No imputation needed — no missing values",
                "Missing_Before": 0, "Filled": 0, "Missing_After": 0
            })
            continue
        modes = result[col].mode(dropna=True)
        if modes.empty:
            strategy_rows.append({
                "Column": col, "Strategy": "Not imputed — no non-missing mode available",
                "Missing_Before": missing_count, "Filled": 0,
                "Missing_After": missing_count
            })
            continue
        result[col] = result[col].fillna(modes.iloc[0])
        strategy_rows.append({
            "Column": col, "Strategy": "Mode",
            "Imputation_Value": modes.iloc[0],
            "Missing_Before": missing_count,
            "Filled": int(missing_count - result[col].isna().sum()),
            "Missing_After": int(result[col].isna().sum())
        })

    after_metrics = calculate_quality_metrics(result)
    missing_after = result.isna().sum()
    strategy_report = pd.DataFrame(strategy_rows)
    missing_comparison = pd.DataFrame({
        "Missing_Before": missing_before,
        "Missing_After": missing_after,
        "Values_Filled": missing_before - missing_after,
        "Percentage_Before": (missing_before / len(data) * 100).round(2),
        "Percentage_After": (missing_after / len(result) * 100).round(2)
    })
    quality_comparison = pd.DataFrame({
        "Metric": list(before_metrics.keys()),
        "Before_Imputation": list(before_metrics.values()),
        "After_Imputation": list(after_metrics.values())
    })
    return result, strategy_report, missing_comparison, quality_comparison, description_details

cleaned_df, strategy_report, missing_comparison, quality_comparison, description_details = intelligent_imputation(deduplicated_df)

print("IMPUTATION STRATEGIES")
display(strategy_report)
print("MISSING VALUES — BEFORE VS AFTER")
display(missing_comparison)
print("DATA QUALITY — BEFORE VS AFTER IMPUTATION")
display(quality_comparison)
print("DESCRIPTION MAPPING DETAILS")
display(pd.DataFrame([description_details]))

# This dataset has no missing Quantity or UnitPrice values; no numeric imputation is expected.
assert cleaned_df["Quantity"].isna().sum() == deduplicated_df["Quantity"].isna().sum()
assert cleaned_df["UnitPrice"].isna().sum() == deduplicated_df["UnitPrice"].isna().sum()
assert cleaned_df["CustomerID"].isna().sum() == deduplicated_df["CustomerID"].isna().sum()
assert cleaned_df["Description"].isna().sum() <= deduplicated_df["Description"].isna().sum()
print("Imputation validation passed; unresolved identifiers/descriptions were not fabricated.")

IMPUTATION STRATEGIES


,Column,Strategy,Missing_Before,Filled,Missing_After,Skewness
0,CustomerID,Not imputed — identifier; missing values retained,135037,0,135037,NaN
1,Quantity,No imputation needed — no missing values,0,0,0,-0.264
2,UnitPrice,No imputation needed — no missing values,0,0,0,185.600
3,Description,StockCode mapping only when mapping is unambig...,1454,1033,421,NaN
4,Country,No imputation needed — no missing values,0,0,0,NaN


MISSING VALUES — BEFORE VS AFTER


,Missing_Before,Missing_After,Values_Filled,Percentage_Before,Percentage_After
InvoiceNo,0,0,0,0.00,0.00
StockCode,0,0,0,0.00,0.00
Description,1454,421,1033,0.27,0.08
Quantity,0,0,0,0.00,0.00
InvoiceDate,0,0,0,0.00,0.00
UnitPrice,0,0,0,0.00,0.00
CustomerID,135037,135037,0,25.16,25.16
Country,0,0,0,0.00,0.00


DATA QUALITY — BEFORE VS AFTER IMPUTATION


,Metric,Before_Imputation,After_Imputation
0,Rows,536641.00,536641.00
1,Columns,8.00,8.00
2,Missing_Cells,136491.00,135458.00
3,Missing_Cell_Percentage,3.18,3.16
4,Exact_Duplicates,0.00,0.00
5,IQR_Outlier_Flags_Quantity_UnitPrice,97951.00,97951.00


DESCRIPTION MAPPING DETAILS


,Missing_Before,Filled_From_StockCode,Remaining_Missing,Conflicting_StockCodes_Not_Imputed
0,1454,1033,421,309


Imputation validation passed; unresolved identifiers/descriptions were not fabricated.


## 7. Data type optimization

The optimizer preserves the semantic role of each field:

- `InvoiceNo`, `StockCode`, `Description`, and `Country` may become unordered `category` columns **only when that uses less memory**.
- `CustomerID` remains a nullable integer identifier (`Int32`), preserving missing values.
- `Quantity` is safely integer-downcast according to its actual minimum and maximum.
- `UnitPrice` remains `float64` to avoid unnecessary precision loss in a financial field.
- `InvoiceDate` remains datetime.

### AI prompt for this code

> Write an `optimize_data_types(data)` function to reduce the memory footprint of the cleaned Online Retail DataFrame without changing its business meaning or values. Preserve `CustomerID` as a nullable integer identifier, downcast `Quantity` only to an integer type that safely contains its full observed range, keep `UnitPrice` as `float64`, and preserve `InvoiceDate` as datetime. For nominal text fields (`InvoiceNo`, `StockCode`, `Description`, and `Country`), test conversion to pandas `category` and apply it only when measured memory usage is lower. Return both the optimized DataFrame and a conversion report explaining each decision. Then measure memory before and after with `memory_usage(deep=True)`, calculate MB saved and percentage reduction, and display type and memory comparison reports. Do not drop rows, round prices, or change values.

In [7]:
def optimize_data_types(data):
    optimized = data.copy(deep=True)
    conversion_rows = []

    # CustomerID is a nominal identifier, not a quantitative measure.
    if "CustomerID" in optimized.columns:
        old_type = str(optimized["CustomerID"].dtype)
        customer_values = pd.to_numeric(optimized["CustomerID"], errors="coerce")
        non_missing = customer_values.dropna()
        if not np.allclose(non_missing, np.round(non_missing)):
            raise ValueError("CustomerID is not integer-like; cannot safely cast to Int32.")
        optimized["CustomerID"] = customer_values.astype("Int32")
        conversion_rows.append({"Column": "CustomerID", "Before": old_type, "After": str(optimized["CustomerID"].dtype), "Reason": "Nullable integer identifier; missing values preserved"})

    # Quantity can be downcast, but the selected integer type must contain its full range.
    if "Quantity" in optimized.columns:
        old_type = str(optimized["Quantity"].dtype)
        optimized["Quantity"] = pd.to_numeric(optimized["Quantity"], downcast="integer")
        conversion_rows.append({"Column": "Quantity", "Before": old_type, "After": str(optimized["Quantity"].dtype), "Reason": "Safe integer downcast based on min/max"})

    # Preserve float64 precision for UnitPrice.
    if "UnitPrice" in optimized.columns:
        old_type = str(optimized["UnitPrice"].dtype)
        optimized["UnitPrice"] = pd.to_numeric(optimized["UnitPrice"], errors="raise").astype("float64")
        conversion_rows.append({"Column": "UnitPrice", "Before": old_type, "After": str(optimized["UnitPrice"].dtype), "Reason": "Financial value; keep float64"})

    # Keep date information as datetime.
    if "InvoiceDate" in optimized.columns:
        old_type = str(optimized["InvoiceDate"].dtype)
        optimized["InvoiceDate"] = pd.to_datetime(optimized["InvoiceDate"], errors="raise")
        conversion_rows.append({"Column": "InvoiceDate", "Before": old_type, "After": str(optimized["InvoiceDate"].dtype), "Reason": "Date/time field"})

    # Convert nominal text columns to category only if measured memory usage improves.
    nominal_text_columns = [c for c in ["InvoiceNo", "StockCode", "Description", "Country"] if c in optimized.columns]
    for col in nominal_text_columns:
        old_type = str(optimized[col].dtype)
        memory_before = optimized[col].memory_usage(deep=True)
        candidate = optimized[col].astype("category")
        memory_after = candidate.memory_usage(deep=True)
        if memory_after < memory_before:
            optimized[col] = candidate
            conversion_rows.append({"Column": col, "Before": old_type, "After": str(optimized[col].dtype), "Reason": f"Categorical encoding reduced memory ({memory_before:,} -> {memory_after:,} bytes)"})
        else:
            conversion_rows.append({"Column": col, "Before": old_type, "After": old_type, "Reason": "Kept original type; category would not reduce memory"})

    return optimized, pd.DataFrame(conversion_rows)

memory_before_mb = cleaned_df.memory_usage(deep=True).sum() / (1024 ** 2)
df_type_optimized, type_conversion_report = optimize_data_types(cleaned_df)
memory_after_mb = df_type_optimized.memory_usage(deep=True).sum() / (1024 ** 2)
memory_saved_mb = memory_before_mb - memory_after_mb
memory_saved_pct = memory_saved_mb / memory_before_mb * 100 if memory_before_mb else 0

print("DATA TYPE CONVERSION REPORT")
display(type_conversion_report)

type_comparison = pd.DataFrame({
    "Before Optimization": cleaned_df.dtypes.astype(str),
    "After Optimization": df_type_optimized.dtypes.astype(str)
})
print("DATA TYPES — BEFORE VS AFTER")
display(type_comparison)

memory_report = pd.DataFrame({
    "Metric": ["Memory Before (MB)", "Memory After (MB)", "Memory Saved (MB)", "Memory Reduction (%)"],
    "Value": [round(memory_before_mb, 2), round(memory_after_mb, 2), round(memory_saved_mb, 2), round(memory_saved_pct, 2)]
})
print("MEMORY OPTIMIZATION")
display(memory_report)

DATA TYPE CONVERSION REPORT


,Column,Before,After,Reason
0,CustomerID,Int32,Int32,Nullable integer identifier; missing values pr...
1,Quantity,int64,int32,Safe integer downcast based on min/max
2,UnitPrice,float64,float64,Financial value; keep float64
3,InvoiceDate,datetime64[us],datetime64[us],Date/time field
4,InvoiceNo,object,category,"Categorical encoding reduced memory (40,705,61..."
5,StockCode,object,category,"Categorical encoding reduced memory (41,548,00..."
6,Description,object,category,"Categorical encoding reduced memory (61,774,19..."
7,Country,str,category,"Categorical encoding reduced memory (54,672,43..."


DATA TYPES — BEFORE VS AFTER


,Before Optimization,After Optimization
InvoiceNo,object,category
StockCode,object,category
Description,object,category
Quantity,int64,int32
InvoiceDate,datetime64[us],datetime64[us]
UnitPrice,float64,float64
CustomerID,Int32,Int32
Country,str,category


MEMORY OPTIMIZATION


,Metric,Value
0,Memory Before (MB),143.68
1,Memory After (MB),39.29
2,Memory Saved (MB),104.39
3,Memory Reduction (%),72.66


## 8. Final validation and export

### AI prompt for this code

> Create a final validation and reporting cell for the cleaned and optimized Online Retail DataFrame. Verify that optimization preserved the DataFrame shape, missing-value counts, duplicate count, and all column values (use appropriate comparisons or stable hashes for category/string columns). Explicitly check that cancellation invoice markers are unchanged, missing `CustomerID` values remain missing, `UnitPrice` is still `float64`, and `InvoiceDate` remains datetime. Build and display a final summary containing the original row count, post-deduplication row count, duplicates removed, remaining missing identifiers/descriptions, and memory before/after optimization and percentage reduction. Export the compact quality and optimization reports into one Excel workbook with clearly named sheets. Keep full cleaned-data CSV and Excel exports optional and disabled by default, because the dataset is large. Finish with clear validation messages.

In [8]:
# Confirm that optimization has not changed row count, column count, values or missingness.
assert df_type_optimized.shape == cleaned_df.shape
assert df_type_optimized.isna().sum().equals(cleaned_df.isna().sum())
assert int(df_type_optimized.duplicated().sum()) == int(cleaned_df.duplicated().sum())

# Verify every value without allocating large temporary object copies.
for column in cleaned_df.columns:
    if column == "Quantity":
        unchanged = np.array_equal(cleaned_df[column].to_numpy(), df_type_optimized[column].to_numpy())
    elif column == "UnitPrice":
        unchanged = np.array_equal(cleaned_df[column].to_numpy(), df_type_optimized[column].to_numpy())
    elif column == "CustomerID":
        unchanged = cleaned_df[column].equals(df_type_optimized[column])
    elif column == "InvoiceDate":
        unchanged = cleaned_df[column].equals(df_type_optimized[column])
    else:
        before_hash = pd.util.hash_pandas_object(cleaned_df[column], index=False)
        after_hash = pd.util.hash_pandas_object(df_type_optimized[column], index=False)
        unchanged = before_hash.equals(after_hash)
    assert unchanged, f"Values changed during optimization: {column}"

# Explicit checks for domain-sensitive fields.
assert (df_type_optimized["InvoiceNo"].astype("string").str.startswith("C", na=False).sum()
        == cleaned_df["InvoiceNo"].astype("string").str.startswith("C", na=False).sum())
assert df_type_optimized["CustomerID"].isna().sum() == cleaned_df["CustomerID"].isna().sum()
assert df_type_optimized["UnitPrice"].dtype == "float64"
assert pd.api.types.is_datetime64_any_dtype(df_type_optimized["InvoiceDate"])

final_summary = pd.DataFrame({
    "Metric": [
        "Original rows", "Rows after exact duplicate removal", "Exact duplicate rows removed",
        "Remaining exact duplicates", "CustomerID missing values retained",
        "Description missing values remaining", "Memory before optimization (MB)",
        "Memory after optimization (MB)", "Memory reduction (%)"
    ],
    "Value": [
        len(df), len(deduplicated_df), len(df) - len(deduplicated_df),
        int(df_type_optimized.duplicated().sum()), int(df_type_optimized["CustomerID"].isna().sum()),
        int(df_type_optimized["Description"].isna().sum()), round(memory_before_mb, 2),
        round(memory_after_mb, 2), round(memory_saved_pct, 2)
    ]
})

display(final_summary)
print("Final optimized dataset preview:")
display(df_type_optimized.head())
print("All final validation checks passed.")

# Export compact reports to Excel.
EXPORT_REPORTS = True
if EXPORT_REPORTS:
    with pd.ExcelWriter("Online_Retail_Data_Cleaning_Reports.xlsx", engine="openpyxl") as writer:
        missing_report.to_excel(writer, sheet_name="Initial Missing Values")
        duplicate_summary.to_excel(writer, sheet_name="Duplicate Summary", index=False)
        outlier_report.to_excel(writer, sheet_name="IQR Outliers", index=False)
        business_rule_report.to_excel(writer, sheet_name="Business Checks", index=False)
        summary_report.to_excel(writer, sheet_name="Initial Summary", index=False)
        strategy_report.to_excel(writer, sheet_name="Imputation Strategy", index=False)
        missing_comparison.to_excel(writer, sheet_name="Missing Comparison")
        quality_comparison.to_excel(writer, sheet_name="Quality Comparison", index=False)
        type_conversion_report.to_excel(writer, sheet_name="Type Conversions", index=False)
        memory_report.to_excel(writer, sheet_name="Memory Report", index=False)
        final_summary.to_excel(writer, sheet_name="Final Summary", index=False)
    print("Exported: Online_Retail_Data_Cleaning_Reports.xlsx")

# Optional: export the full optimized dataset. Disabled by default because
# writing 500k+ rows to Excel/CSV can take extra time and disk space.
EXPORT_CLEANED_DATA_TO_CSV = False
EXPORT_CLEANED_DATA_TO_EXCEL = False

if EXPORT_CLEANED_DATA_TO_CSV:
    df_type_optimized.to_csv("Online_Retail_Cleaned_Optimized.csv", index=False)
    print("Exported: Online_Retail_Cleaned_Optimized.csv")

if EXPORT_CLEANED_DATA_TO_EXCEL:
    df_type_optimized.to_excel("Online_Retail_Cleaned_Optimized.xlsx", index=False)
    print("Exported: Online_Retail_Cleaned_Optimized.xlsx")


,Metric,Value
0,Original rows,541909.00
1,Rows after exact duplicate removal,536641.00
2,Exact duplicate rows removed,5268.00
3,Remaining exact duplicates,0.00
4,CustomerID missing values retained,135037.00
5,Description missing values remaining,421.00
6,Memory before optimization (MB),143.68
7,Memory after optimization (MB),39.29
8,Memory reduction (%),72.66


Final optimized dataset preview:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


All final validation checks passed.
Exported: Online_Retail_Data_Cleaning_Reports.xlsx


## 9. Interpretation and limitations

- Missing `CustomerID` values remain missing because the source does not identify those customers. Do not describe the dataset as having zero missing values.
- Remaining missing `Description` values are those that could not be inferred unambiguously from `StockCode`.
- Negative `Quantity`, cancellation invoices, zero-price lines, and exceptional negative-price adjustment rows are retained. Investigate them before deciding whether to exclude them for a specific business analysis.
- IQR flags are statistical review indicators, not proof of data errors.
- The exact duplicate count applies to fully identical rows across all eight columns; repeated invoice/product combinations are not treated as exact duplicates.